# Preprocessing review

Visual and numeric check of the preprocessing: the breast crop box on the original
images, the preprocessed images (crop, rescale, flip) with their masks, and the
mapping of a mask back onto the original image. Run
`python -m mammo_seg.data.preprocess_dataset` first. Outputs are stripped before
commit (`nbstripout`).

The questions: does every box contain the whole breast and little else, is every
chest wall on the left after preprocessing, and does each mask still sit on its
lesion?


In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pydicom
from scipy import ndimage

from mammo_seg.data.manifest import read_manifest
from mammo_seg.data.masks import mask_path, read_mask
from mammo_seg.data.paths import INBREAST_PROCESSED_DIR, INBREAST_RAW_DIR
from mammo_seg.data.preprocess_dataset import (
    image_path,
    processed_mask_path,
    read_image,
    read_record,
)
from mammo_seg.data.preprocessing import restore_mask


def find_repo_root(start: Path) -> Path:
    for parent in [start, *start.parents]:
        if (parent / "pyproject.toml").exists():
            return parent
    raise FileNotFoundError("pyproject.toml not found above " + str(start))


ROOT = find_repo_root(Path.cwd().resolve())
RAW = ROOT / INBREAST_RAW_DIR
MASKS = ROOT / INBREAST_PROCESSED_DIR / "masks"
OUT = ROOT / INBREAST_PROCESSED_DIR / "preprocessed"

manifest = read_manifest(ROOT / INBREAST_PROCESSED_DIR / "manifest.json")
record = read_record(OUT / "preprocessing.json")
print(record.settings)
print(len(manifest.images), "images,", len(record.images), "preprocessed")

## Crop boxes on the original images

The green rectangle is the crop box (with its margin) on the original DICOM,
downsampled for display. `PAGE` selects 30 images at a time.


In [ ]:
PER_PAGE = 30
STEP = 12  # display downscale of the originals


def show_boxes(images: list) -> None:
    columns = 6
    rows = -(-len(images) // columns)
    fig, axes = plt.subplots(
        rows, columns, figsize=(2.6 * columns, 3.6 * rows), squeeze=False
    )
    for ax in axes.ravel():
        ax.axis("off")
    for ax, image in zip(axes.ravel(), images, strict=False):
        pixels = pydicom.dcmread(RAW / image.dicom_path).pixel_array
        small = pixels[::STEP, ::STEP].astype(np.float32)
        ax.imshow(np.clip(small / 3000, 0, 1), cmap="gray")
        box = record.images[image.file_id].box
        top, bottom, left, right = (
            v / STEP for v in (box.top, box.bottom, box.left, box.right)
        )
        ax.plot(
            [left, right, right, left, left],
            [top, top, bottom, bottom, top],
            color="lime",
            linewidth=1,
        )
        ax.set_title(f"{image.file_id} {image.laterality}{image.view}", fontsize=7)
    plt.tight_layout()
    plt.show()


PAGE = 0
pages = -(-len(manifest.images) // PER_PAGE)
print(f"page {PAGE + 1} of {pages}")
show_boxes(list(manifest.images[PAGE * PER_PAGE : (PAGE + 1) * PER_PAGE]))

### Show every page

Long output; clear it afterwards.


In [ ]:
for page in range(pages):
    show_boxes(list(manifest.images[page * PER_PAGE : (page + 1) * PER_PAGE]))

## Preprocessed images with their masks

The stored 16-bit images, as the model will see them before normalization, with
the mask outline in red. The chest wall should be on the left in every image.


In [ ]:
def show_processed(images: list) -> None:
    columns = 6
    rows = -(-len(images) // columns)
    fig, axes = plt.subplots(
        rows, columns, figsize=(2.6 * columns, 4.2 * rows), squeeze=False
    )
    for ax in axes.ravel():
        ax.axis("off")
    for ax, image in zip(axes.ravel(), images, strict=False):
        pixels = read_image(image_path(OUT, image.file_id))
        mask = read_mask(processed_mask_path(OUT, image.file_id))
        ax.imshow(np.clip(pixels / 3000, 0, 1), cmap="gray")
        if mask.any():
            ax.contour(mask, levels=[0.5], colors="red", linewidths=0.8)
        flipped = "flipped" if record.images[image.file_id].flipped else ""
        ax.set_title(
            f"{image.file_id} {image.laterality}{image.view} {flipped}", fontsize=7
        )
    plt.tight_layout()
    plt.show()


PAGE = 0
print(f"page {PAGE + 1} of {pages}")
show_processed(list(manifest.images[PAGE * PER_PAGE : (PAGE + 1) * PER_PAGE]))

### Show every page

Long output; clear it afterwards.


In [ ]:
for page in range(pages):
    show_processed(list(manifest.images[page * PER_PAGE : (page + 1) * PER_PAGE]))

## Numeric checks on all images

- **Orientation:** after preprocessing, the left third of the image should hold
  more tissue than the right third (chest wall on the left).
- **Box:** the share of the cropped image that is tissue, and how close the tissue
  comes to the box edges (it should reach them on the chest-wall side).
- **Masks:** every mask is brighter than a ring of tissue around it, and a mask
  restored onto the original image overlaps the original mask.


In [ ]:
chest_left = []
tissue_share = []
sizes = []
for image in manifest.images:
    pixels = read_image(image_path(OUT, image.file_id))
    tissue = pixels > 100
    third = pixels.shape[1] // 3
    chest_left.append(tissue[:, :third].mean() > tissue[:, -third:].mean())
    tissue_share.append(tissue.mean())
    sizes.append(pixels.shape)

print(f"chest wall on the left: {sum(chest_left)} of {len(chest_left)} images")
print(
    "not on the left:",
    [m.file_id for m, ok in zip(manifest.images, chest_left, strict=True) if not ok],
)
print(
    f"tissue share of the preprocessed image: median {np.median(tissue_share):.2f}, "
    f"min {min(tissue_share):.2f}, max {max(tissue_share):.2f}"
)
print(
    "longer side:",
    sorted({max(s) for s in sizes}),
    "| shorter side:",
    min(min(s) for s in sizes),
    "to",
    max(min(s) for s in sizes),
)

In [ ]:
def contrast(pixels: np.ndarray, mask: np.ndarray) -> float:
    inside = mask > 0
    ring = ndimage.binary_dilation(inside, iterations=15)
    ring &= ~ndimage.binary_dilation(inside, iterations=3)
    ring &= pixels > 0
    return float(pixels[inside].mean() - pixels[ring].mean())


brighter = []
overlaps = []
for image in manifest.images:
    if not image.has_mass:
        continue
    pixels = read_image(image_path(OUT, image.file_id)).astype(np.float32)
    mask = read_mask(processed_mask_path(OUT, image.file_id))
    brighter.append(contrast(pixels, mask) > 0)
    params = record.images[image.file_id]
    restored = restore_mask(mask, params)
    original = read_mask(mask_path(MASKS, image.file_id))
    overlaps.append(
        ((restored == 1) & (original == 1)).sum()
        / ((restored == 1) | (original == 1)).sum()
    )

print(f"mask brighter than its surroundings: {sum(brighter)} of {len(brighter)} images")
print(
    f"restored mask vs original mask (IoU): min {min(overlaps):.3f}, "
    f"median {np.median(overlaps):.3f}"
)